In [1]:
import json
import math
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from torch import nn
from torch.nn import functional as F

# training.ipynb vive en experimentation/, y el paquete `modules` quedó en la raíz del proyecto
project_root = Path.cwd().resolve().parent
sys.path.append(str(project_root))

from modules.visualizations import visualize_attention, visualize_attention_on_images
from modules.model import ViTForClassfication
from modules.datasets import CelebADataset, CIFAR10Dataset
from modules.effective_rank import effective_rank
import torch
from torch import nn, optim
from modules.experiments import save_experiment, save_checkpoint
import logging
from rich.logging import RichHandler
from tqdm.auto import tqdm

# Apuntar siempre al mlflow.db local (Jupyter ejecuta desde el directorio del notebook)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Logging con colores vía rich: nivel, timestamp y mensaje coloreados por severidad.
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(rich_tracebacks=True, show_path=False)],
    force=True,
)
logger = logging.getLogger("vit_training")

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
dataset = CIFAR10Dataset()
# logger.info(f"Cargando dataset {dataset.name} desde {dataset.root}...")
trainloader, testloader, classes = dataset.get_loaders(batch_size=128, num_workers=4)
logger.info(f"Dataset {dataset.name} listo: {len(dataset.trainset)} train / {len(dataset.testset)} test, {dataset.n_classes} clases.")

In [ ]:
dataset

In [ ]:
INDICES = [10, 12, 34, 46, 100]

In [ ]:
dataset.visualize_sample(indices=INDICES, set="train")

In [ ]:
config = {
    "patch_size": 4,  # Input image size: 128x128 -> 32x32 patches
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads":4,
    "intermediate_size": 4 * 48, # 4 * hidden_size
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 32,
    "num_classes": dataset.n_classes,  # 40 atributos binarios de CelebA (multi-label)
    "num_channels": 3,
    "qkv_bias": True,
    "use_faster_attention": False,
}

model = ViTForClassfication(config).to(device)


In [ ]:
dataset.get_samples_from_indices(INDICES)[0].shape

In [ ]:
random_input = dataset.get_samples_from_indices(INDICES)[0].to(device)
logits, attention_maps = model(random_input, output_attentions=True)

In [ ]:
attention_maps[0].shape

In [ ]:
visualize_attention(attention_maps[0])

In [ ]:
attention_maps[0].shape

In [ ]:
dataset.get_samples_from_indices(INDICES, set="training")[0].shape

In [ ]:
visualize_attention_on_images(attention_maps[0], dataset.get_samples_from_indices(INDICES, set='training')[0])[0]

In [ ]:
is_interpretable = False #@param {type:"boolean"}
epochs = 10 #@param {type: "integer"}
exp_name = f'vit-with-{epochs}-epochs{"-interpretable" if is_interpretable else ""}-{dataset.name}' #@param {type:"string"}
batch_size = 32 #@param {type: "integer"}
lr = 1e-2  #@param {type: "number"}
save_model_every = 0 #@param {type: "integer"}
lambda_loss = 0.1 #@param {type: "number"}

mlflow.set_experiment(exp_name)

config = {
    "patch_size": 4,  # Input image size: 128x128 -> 32x32 patches
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads": 4,
    "intermediate_size": 4 * 48, # 4 * hidden_size
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 32,
    "num_classes": dataset.n_classes,  # 40 atributos binarios de CelebA (multi-label)
    "num_channels": 3,
    "qkv_bias": True,
    "use_faster_attention": False,
}
assert config["hidden_size"] % config["num_attention_heads"] == 0
assert config['intermediate_size'] == 4 * config['hidden_size']
assert config['image_size'] % config['patch_size'] == 0

In [ ]:
def interpretability_loss(attention_maps):
    # attention_maps: (B, H, P) — atención del CLS hacia los parches por cada cabeza

    B, H, P = attention_maps.shape

    # Normalización L2 antes del producto punto entre cabezas.
    # Así la similitud queda en [-1, 1] (similitud coseno) y no depende de la magnitud
    # absoluta del vector de atención — evita que el modelo reduzca la loss
    # simplemente achicando los valores de atención.
    attention_maps = F.normalize(attention_maps, p=2, dim=-1)

    # Producto punto entre todos los pares de cabezas (i, j) posición a posición
    mm = torch.einsum('bhp,bkp->bhkp', attention_maps, attention_maps)

    # Máscara [H, H] → True en diagonal (no comparar cabeza consigo misma)
    diag_mask = torch.eye(H, dtype=torch.bool, device=attention_maps.device)

    # Anular la diagonal para que no contribuya a la loss
    mm_masked = mm.clone()
    mm_masked[:, diag_mask, :] = float(0)

    mm_sum = mm_masked.sum(dim=(1, 2))   # (B, P)
    mm_total = mm_sum.sum(dim=1)          # (B,)  una pérdida por imagen

    # Reducir a escalar para que .backward() funcione.
    # mean() mantiene la magnitud del gradiente independiente del batch size.
    return mm_total.mean()

In [ ]:
class Trainer:
    """
    The simple trainer.
    """

    def __init__(self, model, optimizer, loss_fn, exp_name, device, dataset=None, log_indices=None):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.exp_name = exp_name
        self.device = device
        self.dataset = dataset
        self.log_indices = log_indices
        self.multi_label = getattr(dataset, "multi_label", False) if dataset is not None else False

    def train(self, trainloader, testloader, epochs, save_model_every_n_epochs=0):
        list_train_diversity_losses = []
        list_train_losses = []
        list_test_losses = []
        list_test_diversity_losses = []

        logger.info(f"Iniciando entrenamiento '{self.exp_name}': {epochs} épocas, "
                    f"batch_size={batch_size}, lr={lr}, device={self.device}")

        with mlflow.start_run(run_name="run-experiment"):
            mlflow.log_param("epochs", epochs)
            mlflow.log_param("batch_size", batch_size)
            mlflow.log_param("learning_rate", lr)
            mlflow.log_param("lambda_loss", lambda_loss)

            epoch_bar = tqdm(range(epochs), desc="Épocas", unit="epoch")
            for i in epoch_bar:
                train_losses = self.train_epoch(trainloader, epoch_i=i)
                test_losses = self.evaluate(testloader, epoch_i=i)

                epoch_bar.set_postfix(
                    train_loss=f"{train_losses['total_loss']:.4f}",
                    test_loss=f"{test_losses['total_loss']:.4f}",
                    test_acc=f"{test_losses['accuracy']:.4f}",
                )

                logger.info(f"Epoch {i+1}/{epochs} - Train Loss: {train_losses['total_loss']:.4f}, "
                            f"Train Diversity Loss: {train_losses['interpretability_loss']:.4f}, "
                            f"Test Loss: {test_losses['total_loss']:.4f}, "
                            f"Test Accuracy: {test_losses['accuracy']:.4f}")

                if save_model_every_n_epochs > 0 and (i+1) % save_model_every_n_epochs == 0 and i+1 != epochs:
                    logger.info(f"Guardando checkpoint en época {i+1}")
                    save_checkpoint(self.exp_name, self.model, i+1)

        save_experiment(self.exp_name, config, self.model, test_losses, base_dir="experiments")
        logger.info(f"Entrenamiento finalizado. Modelo y métricas guardados para '{self.exp_name}'.")

    def train_epoch(self, trainloader, epoch_i=0):
        self.model.train()
        losses = {
            "cls_loss": [],
            "interpretability_loss": [],
            "total_loss": [],
        }

        batch_bar = tqdm(trainloader, desc=f"Época {epoch_i+1} [train]", unit="batch", leave=False)
        for batch in batch_bar:
            batch = [t.to(self.device) for t in batch]
            images, labels = batch
            self.optimizer.zero_grad()

            logits, attention_maps = self.model(images, output_attentions=True)

            interpretability_loss_value = interpretability_loss(attention_maps[-1][:, :, 0, 1:])
            # BCEWithLogitsLoss (multi-label) espera targets float; CrossEntropyLoss (single-label) espera índices long.
            cls_loss = self.loss_fn(logits, labels.float() if self.multi_label else labels)

            if is_interpretable:
                total_loss = (1 - lambda_loss) * cls_loss + lambda_loss * interpretability_loss_value
            else:
                total_loss = cls_loss

            total_loss.backward()
            self.optimizer.step()

            losses["cls_loss"].append(cls_loss.item())
            losses["interpretability_loss"].append(interpretability_loss_value.item())
            losses["total_loss"].append(total_loss.item())

            batch_bar.set_postfix(loss=f"{total_loss.item():.4f}")

        avg_cls_loss = sum(losses["cls_loss"]) / len(losses["cls_loss"])
        avg_interpretability_loss = sum(losses["interpretability_loss"]) / len(losses["interpretability_loss"])
        avg_total_loss = sum(losses["total_loss"]) / len(losses["total_loss"])

        logger.debug(f"Época {epoch_i+1} [train] - cls_loss={avg_cls_loss:.4f}, "
                     f"interpretability_loss={avg_interpretability_loss:.4f}, total_loss={avg_total_loss:.4f}")

        mlflow.log_metric("loss", avg_cls_loss, step=epoch_i)
        mlflow.log_metric("interpretability_loss", avg_interpretability_loss, step=epoch_i)
        mlflow.log_metric("total_loss", avg_total_loss, step=epoch_i)

        return {
            "cls_loss": avg_cls_loss,
            "interpretability_loss": avg_interpretability_loss,
            "total_loss": avg_total_loss
        }

    @torch.no_grad()
    def evaluate(self, testloader, epoch_i=0):
        self.model.eval()

        losses = {
            "cls_loss": [],
            "interpretability_loss": [],
            "total_loss": [],
        }

        num_classes = config["num_classes"]
        total_correct = 0
        total_samples = 0

        if self.multi_label:
            # Precision por atributo: cuenta de aciertos por columna, acumulada a lo largo de todo el testset.
            attr_correct = torch.zeros(num_classes, dtype=torch.long)
        else:
            # Matriz de confusión acumulada en CPU para no transferir tensores por batch.
            confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)

        batch_bar = tqdm(testloader, desc=f"Época {epoch_i+1} [eval]", unit="batch", leave=False)
        for batch in batch_bar:
            batch = [t.to(self.device) for t in batch]
            images, labels = batch

            logits, attention_maps = self.model(images, output_attentions=True)
            # Misma señal que en train: atención del CLS (pos 0) hacia los parches (1:) en la última capa
            interpretability_loss_value = interpretability_loss(attention_maps[-1][:, :, 0, 1:])
            cls_loss = self.loss_fn(logits, labels.float() if self.multi_label else labels)

            if is_interpretable:
                total_loss = (1 - lambda_loss) * cls_loss + lambda_loss * interpretability_loss_value
            else:
                total_loss = cls_loss

            losses["cls_loss"].append(cls_loss.item())
            losses["interpretability_loss"].append(interpretability_loss_value.item())
            losses["total_loss"].append(total_loss.item())

            if self.multi_label:
                # Cada una de las 40 salidas es un sigmoide binario independiente (atributo presente/ausente).
                preds = (torch.sigmoid(logits) > 0.5).long()
                matches = (preds == labels)
                total_correct += matches.sum().item()
                total_samples += labels.numel()  # accuracy promediada sobre todos los (sample, atributo)
                attr_correct += matches.sum(dim=0).cpu()
            else:
                preds = logits.argmax(dim=-1)

                total_correct += (preds == labels).sum().item()
                total_samples += labels.size(0)

                # Matriz de confusión acumulada vía bincount (O(N) por batch, sin loop Python).
                # Truco: index = true * num_classes + pred → bin único por celda (true, pred).
                idx = labels * num_classes + preds
                binc = torch.bincount(idx, minlength=num_classes * num_classes).cpu()
                confusion += binc.view(num_classes, num_classes)

            batch_bar.set_postfix(acc=f"{total_correct / total_samples:.4f}")

        avg_cls_loss = sum(losses["cls_loss"]) / len(losses["cls_loss"])
        avg_interpretability_loss = sum(losses["interpretability_loss"]) / len(losses["interpretability_loss"])
        avg_total_loss = sum(losses["total_loss"]) / len(losses["total_loss"])
        accuracy = total_correct / total_samples

        logger.debug(f"Época {epoch_i+1} [eval] - cls_loss={avg_cls_loss:.4f}, "
                     f"interpretability_loss={avg_interpretability_loss:.4f}, "
                     f"total_loss={avg_total_loss:.4f}, accuracy={accuracy:.4f}")

        mlflow.log_metric("val_loss", avg_cls_loss, step=epoch_i)
        mlflow.log_metric("val_interpretability_loss", avg_interpretability_loss, step=epoch_i)
        mlflow.log_metric("val_total_loss", avg_total_loss, step=epoch_i)
        mlflow.log_metric("val_accuracy", accuracy, step=epoch_i)

        # Rango efectivo (Roy & Vetterli, 2007) de las matrices Q/K/V de cada cabeza
        # de la última capa de atención — señal de qué tan redundante es cada cabeza.
        last_block = self.model.encoder.blocks[-1]
        heads = last_block.attention.heads

        query_weights = torch.stack([head.query.weight.detach() for head in heads])
        key_weights = torch.stack([head.key.weight.detach() for head in heads])
        value_weights = torch.stack([head.value.weight.detach() for head in heads])

        erank_query = effective_rank(query_weights)
        erank_key = effective_rank(key_weights)
        erank_value = effective_rank(value_weights)

        for h in range(len(heads)):
            mlflow.log_metric(f"effective_rank/Q/head_{h}", erank_query[h].item(), step=epoch_i)
            mlflow.log_metric(f"effective_rank/K/head_{h}", erank_key[h].item(), step=epoch_i)
            mlflow.log_metric(f"effective_rank/V/head_{h}", erank_value[h].item(), step=epoch_i)

        mlflow.log_metric("effective_rank/Q/mean", erank_query.mean().item(), step=epoch_i)
        mlflow.log_metric("effective_rank/K/mean", erank_key.mean().item(), step=epoch_i)
        mlflow.log_metric("effective_rank/V/mean", erank_value.mean().item(), step=epoch_i)

        class_names = getattr(self.dataset, "classes", None) if self.dataset is not None else None
        if class_names is None:
            class_names = [str(i) for i in range(num_classes)]

        if self.multi_label:
            # --- Accuracy por atributo (barras) ---
            # Reemplaza la matriz de confusión: con atributos binarios independientes,
            # una matriz clase-contra-clase no tiene sentido; en cambio mostramos qué
            # tan bien predice el modelo cada atributo por separado.
            attr_samples = total_samples / num_classes
            attr_acc = (attr_correct.float() / attr_samples).numpy()

            fig, ax = plt.subplots(figsize=(10, 8))
            order = attr_acc.argsort()
            ax.barh(range(num_classes), attr_acc[order], color="tab:blue")
            ax.set_yticks(range(num_classes))
            ax.set_yticklabels([class_names[i] for i in order], fontsize=7)
            ax.set_xlabel("Accuracy")
            ax.set_xlim(0, 1)
            ax.set_title(f"Accuracy por atributo — epoch {epoch_i}")
            fig.tight_layout()

            mlflow.log_figure(fig, f"attribute_accuracy/epoch_{epoch_i:03d}.png")
            plt.close(fig)
        else:
            # --- Matriz de confusión como imagen ---
            # Normalizamos por fila (true class): cada celda (i, j) es la proporción de la
            # clase i que fue clasificada como j. La diagonal pasa a ser el recall por clase.
            cm = confusion.float()
            cm_norm = cm / cm.sum(dim=1, keepdim=True).clamp(min=1)
            cm_np = cm_norm.numpy()

            fig, ax = plt.subplots(figsize=(8, 7))
            im = ax.imshow(cm_np, cmap="Blues", vmin=0, vmax=1)
            ax.set_title(f"Confusion matrix (row-normalized) — epoch {epoch_i}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_xticks(range(num_classes))
            ax.set_yticks(range(num_classes))
            ax.set_xticklabels(class_names, rotation=45, ha="right")
            ax.set_yticklabels(class_names)

            # Anotar cada celda con su proporción
            for i in range(num_classes):
                for j in range(num_classes):
                    val = cm_np[i, j]
                    ax.text(j, i, f"{val:.2f}",
                            ha="center", va="center",
                            fontsize=8,
                            color="white" if val > 0.5 else "black")

            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            fig.tight_layout()

            mlflow.log_figure(fig, f"confusion_matrix/epoch_{epoch_i:03d}.png")
            plt.close(fig)

        # Log attention
        if self.dataset is not None and self.log_indices is not None:
            log_images, _ = self.dataset.get_samples_from_indices(self.log_indices, device=self.device, set="train")
            _, log_attention_maps = self.model(log_images, output_attentions=True)

            for layer_idx, amap in enumerate(log_attention_maps):
                attn_img = visualize_attention(amap)
                mlflow.log_image(attn_img, f"attention/layer_{layer_idx}/epoch_{epoch_i:03d}.png")

            overlay_imgs = visualize_attention_on_images(log_attention_maps, log_images)
            for layer_idx, overlay_img in enumerate(overlay_imgs):
                mlflow.log_image(overlay_img, f"attention_overlay/layer_{layer_idx}/epoch_{epoch_i:03d}.png")

        return {
            "cls_loss": avg_cls_loss,
            "interpretability_loss": avg_interpretability_loss,
            "total_loss": avg_total_loss,
            "accuracy": accuracy
        }


In [ ]:
save_model_every_n_epochs = save_model_every
logger.info(f"Recargando loaders con batch_size={batch_size}...")
trainloader, testloader, _ = dataset.get_loaders(batch_size=batch_size, num_workers=4)
model = ViTForClassfication(config)
logger.info(f"Modelo ViT creado: {sum(p.numel() for p in model.parameters()):,} parámetros.")


In [ ]:

optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
# CelebA es multi-label (40 atributos binarios independientes por imagen), no
# clasificación de una sola clase: BCEWithLogitsLoss trata cada salida del
# clasificador como un sigmoide binario independiente.
loss_fn = nn.BCEWithLogitsLoss() if dataset.multi_label else nn.CrossEntropyLoss()
trainer = Trainer(model, optimizer, loss_fn, exp_name, device=device, dataset=dataset, log_indices=INDICES)
trainer.train(trainloader, testloader, epochs, save_model_every_n_epochs=save_model_every_n_epochs)


In [ ]:
mlflow.end_run()

In [ ]:
current_date = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
model_path = f"vit_final_{dataset.name}_{current_date}_{'interpretable' if is_interpretable else 'non_interpretable'}.pt"
torch.save(model.state_dict(), model_path)
logger.info(f"Modelo final guardado en {model_path}")